# Ximenez Line Map (XML TO CSV)

Map Ximenez line breaks with paragraphs.

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
pd.set_option('display.max_colwidth', False)

In [3]:
xml_lines = open("xom-all-flat-mod-pnums.xml").readlines()

In [50]:
els = dict(
    lang = '',
    folio = 0,
    side = 0,
    para = 0,
    lb = 0,
    lb_n  = '' # Line number as given
)

data = []

for line in xml_lines:
    
    # Grab column (lang) 
    # Pattern: <div xml:lang="quc" type="column" rend="left half">
    if re.match(r"<div xml:lang", line):
        els['lang'] = line.split('"')[1].split('"')[0]
        els['para'] = 0

    # Grab manuscript
    if re.match(r"^<pb ", line):
        f, s = line.split("xom-")[1].split('"')[0].split('-')
        els['folio'] = int(f[1:])
        els['side'] = int(s[1:])
        els['lb'] = 0

    # Grab paragraph
    if re.match(r"^<p ", line):
        els['para'] += 1

    # Grab line break
    if re.match(r"^<lb n=", line):
        els['lb_n'] = line.split('<lb n="')[1].split('"')[0]
        els['lb'] += 1
        els['lb_str'] = ' '.join(line.split("/>")[1:]).strip()
        data.append(tuple(els.values()))

In [51]:
LINE = pd.DataFrame(data, columns=els.keys())
LINE = LINE.set_index(['lang','folio', 'side', 'para', 'lb'])

In [52]:
LINE

lb_n  \
lang folio side para lb        
quc  1     1    1    1   1     
                     2   2     
                2    3   3     
                     4   4     
                     5   5     
...                     ..     
spa  56    2    100  31  31    
                     32  32    
                     33  33    
                     34  34    
                     35  35    

                                                                                                                                                                                                                                                                 lb_str  
lang folio side para lb                                                                                                                                                                                                                                                  
quc  1     1    1    1   <hi rend="very-large">ARE V XE OHER</hi>                                                                                                                                                                                                        
                     2   Ꜩih varal Quiche vbi.                                                                                                                                                                                                                           
                2    3   <hi rend="large">V</hi>aral xchicaꜩibah vi xchica<pc> –</pc>                                                                                                                                                                                    
                     4   tiquiba vi oher ꜩih, vticaribal,                                                                                                                                                                                                                
                     5   vxenabal puch ronohel xban,                                                                                                                                                                                                                     
...                                                   ...                                                                                                                                                                                                                
spa  56    2    100  31  quiche</rs>. <choice><abbr>porq’</abbr><expan instant="false">porque</expan></choice> ya no ay <rs ana="POPOL_WUJ">donde leerlo</rs>                                                                                                            
                     32  yaun<choice><abbr>q’</abbr><expan instant="false">que</expan></choice> antíguamente lo auía, pero                                                                                                                                               
                     33  se ha perdído. y aquí seacabo todo lo<pc> –</pc>                                                                                                                                                                                                
                     34  tocante <rs ana="TINAMIT_K'ICHE'">al quiche</rs>, <choice><abbr>q’</abbr><expan instant="false">que</expan></choice> se llama<rs ana="SANTA_CRUZ"> <choice><abbr type="superscription">Sta</abbr><expan instant="false">Santa</expan></choice>  
                     35  Cruz.</rs>                                                                                                                                                                                                                                      

[10177 rows x 2 columns]

In [35]:
assert LINE.index.has_duplicates == False, "LINE has duplicates"

In [36]:
LINE.loc['spa'].sample(5)

,,,,lb_n,lb_str
folio,side,para,lb,,
44,1,69,17,16,"se lababan, y deshonestamente. y se"
26,2,35,33,32,"erno</rs>, <choice><abbr>q’</abbr><expan instant=""false"">que</expan></choice> es esto como no han muer<pc> –</pc>"
24,1,34,11,11,"<choice><abbr>q’</abbr><expan instant=""false"">que</expan></choice> es eso <rs ana=""WUQUB'_KAME'"">vucub came</rs><choice><abbr>q’</abbr><expan instant=""false"">que</expan></choice> te ha mordído le"
37,1,59,42,42,"ron lo<choice><abbr>q’</abbr><expan instant=""false"">que</expan></choice> deçía el <rs ana=""TOJIL"">tohíl</rs>. esta bíen dí<pc> –</pc>"
22,2,32,7,7,"píojo</rs>, y anduvo. y le díxo: <rs ana=""KA'IB'_K'AJOLAB'"">tu mí níeto</rs> que<pc> –</pc>"


In [37]:
LINE.loc['quc'].sample(5)

,,,,lb_n,lb_str
folio,side,para,lb,,
23,2,31,31,31,"hal ha zivan xachu xol <rs ana=""TZ'IKIN"">q<hi rend=""subscript"">’</hi>iquin</rs>"
44,1,66,41,41,chivil qui vach xohohvqhaxic
45,2,68,19,19,hil</rs> are cabauil arepu chica
31,2,42,7,7,"chive ohva, oh <rs ana=""JUNAJPU"">xhun ahpu</rs>"
20,2,27,17,17,cut chicochoch quiꜩih mixohco<pc> –</pc>


In [38]:
LINE['lb_str_plain'] = LINE.lb_str.str.replace(r"<[^>]+/?>", "", regex=True).str.replace(" –", "–", regex=False)

In [39]:
chars = {
    'Ꜩ': 'Tz',
    'ꜩ': 'tz',
    'ꜫ': "q'",
    'ÿ': 'i', # 'ij' 
}

In [40]:
for char in chars:
    LINE.lb_str_plain = LINE.lb_str_plain.str.replace(char, chars[char], regex=False)

In [41]:
LINE.loc['spa'].to_csv("ximenez-spa-LINE.csv", index=True)
LINE.loc['quc'].to_csv("ximenez-quc-LINE.csv", index=True)